# 운영형 에이전트

## 1. Middleware와 Guardrails - PII 가드레일

### 1-1. 왜 Middleware인가?

* 에이전트가 강해질수록 아무때나 자유롭게 실행되면 위험해짐.
    * ex) 이메일 전송, DB쓰기, 파일삭제, 외부 서비스 호출 등 - 모델이 혼자 판단해서 바로 실행하면 안됨

* Middleware가 없으면 이런 규칙을 프롬프트나 Tool 내부 코드에 흩어넣게 됨.
    * 민감정보 처리, 승인 대기, 비용 제한, 실행 로그 같은 공통 규칙 늘어나면 어딜 막고 고쳐야하는지 찾기 어려워짐
* Middleware는 이런 공통 규칙을 실행 흐름의 정해진 지점에 모아둠

[훅]
* before_agent: 에이전트 실행 시작 전(한번) - 요청 초기화, 사용자 권한 확인, 전체 입력점검
* before_model: 모델 호출 직전(매번) - 메시지 압축, context추가, 위험 입력차단
* wrap_model_call: 모델 호출 전체를 감쌈 - 재시도, 예비 모델로 바꾸기, 비용 제한
* after_model: 모델 응답 직후, Tool 실행 전 - Tool 호출 검사, HITL 승인, 응답 필터링
* wrap_tool_call: Tool 실행 전체를 감쌈 - 인자 검증, 실행 로그, 실패 처리, 위험 Tool 차단
* after_agent: 에이전트 실행 종료 후 (한번) - 최종 응답 기록, 사용량 집계, 감사 로그
* dynamic_prompt: 모델 호출 전 시스템 프롬프트 구성 - 사용자 역할/권한에 맞는 지시 추가


==> create_agent의 속이 llm 노드와 tools 노드의 루프임.   
    Middleware 훅은 그 루프의 Edge 위에 끼워 넣은 작은 노드.   
    before_model은 llm 노드로 들어가는 모든 화살표 위, after_model은 llm 노드에서 나가는 화살표 위

* 그래프를 직접 짠다면 노드를 하나 추가하면 될 일을, create_agent에서는 middleware로 함
* 도구 실행 뒤에 검증 노드 끼우기 -> create_agent는 middleware로 우회

### 1-2. Guardrails와 Middleware 관계

* Guardrails = 규칙, Middleware = 규칙을 적용하는 구현 방식

### 1-3. PIIMiddleware - 이메일을 모델에 보내기 전에 가리기
* PII: 개인을 식별할 수 있는 정보. 이메일, 전화번호, 주민등록번호, 주소처럼 그대로 모델에 보내거나 로그를 남기면 위험한 값들

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

pii_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트 입니다."
)

pii_result = pii_agent.invoke(
    {'messages': [{'role': 'user', 'content': '내 이메일은 minsu.teacher@example.com이고 강의 자료 다시 보내줘'}]},
    lf_config
)

print("[마스킹된 입력]")
print(pii_result["messages"][0].content)
print("\n[에이전트 응답]")
print(pii_result["messages"][-1].content)

[strategy]
* redact: [REDACTED_...]로 전체를 바꿈 -> 원문을 남길 필요가 없을 때
* mask: 일부만 남기고 가림 -> 카드 번호처럼 끝자리 확인이 필요할 때
* hash: 같은 값이면 같은 해시로 -> 누구인지는 모르게, "같은 사람인지"만 추적할 때
* block: 요청을 막고 예외 발생 -> 민감정보가 있으면 처리 자체를 중단해야할 때

[옵션]
* apply_to_input (True) - before_model :: 사용자 입력
* apply_to_output (False) - after_model :: 모델 응답(사용자에게 나가기 전)
* apply_to_tool_results (False) - before_model :: Tool 결과 (다음 모델 호출에 들어가기 전)

In [ ]:
# 실습 - 신용카드 번호 부분 마스킹

mask_agent = create_agent(
    model = model,
    tools = [],
    middleware = [
        PIIMiddleware(
            "creadit_card",
            strategy="mask",
            apply_to_input = True
        )
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",
)

mask_result = mask_agent.invoke(
    {"messages": [{"role": "user", "content": "내 법인카드는 5105-1051-0510-5100 이고 결제 내역을 확인해줘."}]},
)

print(mask_result["messages"][0].content)

### 1-4. 커스텀 PII - 정규표현식 detector
* 한국어 입력에서 기본 감지가 놓치는 구멍을 커스텀 detector로 메울 수 있음

In [ ]:
custom_pii_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware(
            "member_id",
            detector=r"CUST-\d{4}-\d{4}",
            strategy="redact",
            apply_to_input=True
        )
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",
)

### 1-5. 직접 Middleware 구현 - wrap_tool_call
* 내장 middleware로 안 되는 처리는 훅에 직접 함수를 붙임

In [1]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def clean_tool_result(request, handler):
    # request: 실행하려는 Tool 호출 정보
    # hanlder: 실제 Tool 실행하는 함수
    result = handler(request)

    if isinstance(result, ToolMessage) and isinstance(result.content, str):
        cleaned = result.content.replace("광고", "")[:1000]

        return ToolMessage(content=cleaned, tool_call_id=result.tool_call_id)

    return result

In [8]:
from langchain.tools import tool
from langchain.agents import create_agent

MODEL = 'gpt-4o-mini'

@tool
def search_course_material(query: str) -> str:
    """강의 자료를 검색합니다. 사용자의 질문에 답하기 전에 호출하세요"""
    return "광고 광고 광고 실제 검색 결과: " + ("미들웨어는 … 공통 규칙을 적용합니다. " * 80)

middleware_agent = create_agent(
    model=MODEL,
    tools=[search_course_material],
    middleware=[clean_tool_result],
    system_prompt=""
)

result = middleware_agent.invoke({"messages": [{'role': 'user', 'content': "LangChain middleware 수업 자료의 핵심을 요약해줘."}]})
tool_messages = [msg for msg in result["messages"] if isinstance(msg, ToolMessage)]
assert tool_messages, "Tool이 호출되지 않았습니다. 모델 지시문을 더 강하게 조정하세요."

[요약]
1. Middleware = 에이전트 루프의 정해진 지점(훅)에 끼우는 제어코드. 
    * Guardrail = 규칙, Middleware = 구현

2. PIIMiddleware: 종류 x 전략(redact/mask/hash/block) x 대상(input/output/toolo_results)

3. 가드레일이 과잉 거절을 부를 수 있다 -> 가려진 값의 의미를 프롬프트로 알려 준다

4. 한국어 조사는 기본 감지를 피해간다 -> 정규식 detector로 보완

5. wrap_tool_call의 handler(request): 부르면 실행, 안 부르면 차단

## 2. HITL - 실행 직전에 사람 승인 끼워 넣기
* 에이전트가 어떤 행동을 바로 실행하지 않고, 중간에 사람의 확인을 받도록 멈추는 구조
* 모델이 Tool을 고를 수 있더라도, 그 실행 권한까지 항상 자동으로 줘야하는 건 아님

### 2-1. 언제 사람을 끼우는지?
* 발송, 삭제, 결제, 승인 같은 되돌리기 어려운 직업
* 법무, 인사, 재무처럼 책임 소재가 중요한 업무
* 사람이 중간 판단을 내려야 하는 승인형 워크플로

In [ ]:
from langchain.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

MODEL = 'gpt-4o-mini'

lf_config = {}

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """지정한 수신자에게 이메일을 보냅니다. 실제 발송이므로 신중하게 사용하세요"""
    return f"이메일 발송 완료: to={to}, subject={subject}"

hitl_agent = create_agent(
    model=MODEL,
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {"allowed_decisions": ["approve", "edit", "reject"]}
            }
        )
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "이메일 발송 요청이 오면 설명하지 말고 send_email 도구를 바로 사용하세요. "
        "사용자 메시지에 수신자, 제목, 본문이 이미 있으면 추가 질문하지 마세요. "
        "사람이 Tool 실행을 거절했다는 메시지를 받으면 그 즉시 같은 Tool 호출을 반복하지 말고 취소되었다고 짧게 답하세요. "
        "이후 사용자가 메일 발송을 새로 요청하면 새로운 요청으로 보고 send_email 도구를 사용하세요."
    ),
)

hitl_config = {"configurable": {"thread_id": "email-approval-demo"}}
hitl_config.update(lf_config)

first_step = hitl_agent.invoke(
    {"messages": [{"role": "user", "content":
        "support-team@example.com으로, 제목은 'ORD-1001 배송 지연 고객 안내', "
        "본문은 '고객지원팀 여러분, … 확인해 주세요.' 내용으로 안내 메일을 보내줘."}]},
    hitl_config,
)

# first_action = first_step["__interrupt__"][0].value["action_requests"][0]
# print({"name": first_action["name"], "args": first_action["args"]})

{'name': 'send_email', 'args': {'to': 'support-team@example.com', 'subject': 'ORD-1001 배송 지연 고객 안내', 'body': '고객지원팀 여러분, … 확인해 주세요.'}}


In [17]:
# first_step["__interrupt__"][0].value["action_requests"][0]
print(first_step["__interrupt__"][0].value["action_requests"][0])

{'name': 'send_email', 'args': {'to': 'support-team@example.com', 'subject': 'ORD-1001 배송 지연 고객 안내', 'body': '고객지원팀 여러분, … 확인해 주세요.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'support-team@example.com', 'subject': 'ORD-1001 배송 지연 고객 안내', 'body': '고객지원팀 여러분, … 확인해 주세요.'}"}


### 2-2. 재개 - Command(resume=...)

In [18]:
from langgraph.types import Command

review_decision = {
    "type": "reject",
    "message": "사용자가 이번 메일 발송을 거절했습니다. 현재 중단된 호출은 취소하고, 취소되었다고 짧게 답하세요.",
}

resumed = hitl_agent.invoke(
    Command(resume={"decisions": [review_decision]}),
    hitl_config,
)

for message in resumed["messages"][-3:]:
    content = message.content or "(내용 없음)"
    print(f"{message.__class__.__name__}: {content}")

AIMessage: (내용 없음)
ToolMessage: User rejected the tool call for `send_email` with reason: 사용자가 이번 메일 발송을 거절했습니다. 현재 중단된 호출은 취소하고, 취소되었다고 짧게 답하세요.
AIMessage: 취소되었습니다.


In [ ]:
# 실습 - 공지 메일 승인 에이전트
@tool
def send_notice_email(to: str, subject: str, body: str) -> str:
    """수신자, 제목, 본문이 정해진 공지 메일을 실제로 발송할 때 사용합니다. 외부 발송이므로 승인 후 실행해야 합니다."""
    return f"공지 메일 발송 완료: to={to}, subject={subject}"

approval_agent = create_agent(
    model=model,
    tools=[send_notice_email],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"send_notice_email": {"allowed_decisions": ["approve", "edit", "reject"]}}
    )],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "공지 메일 발송 요청이 오면 설명하지 말고 send_notice_email 도구를 바로 사용하세요. "
        "사용자 메시지에 수신자, 제목, 본문이 이미 있으면 추가 질문하지 마세요."
    ),
)

approval_config = {"configurable": {"thread_id": "hitl-practice-1"}}
approval_config.update(lf_config)

request_message = {"messages": [{"role": "user", "content": "all-staff@example.com으로 제목은 '10월 정기 점검 안내', 본문은 '10월 3일 오전 9시부터 1시간 동안 사내 시스템 점검이 있습니다.'로 공지 메일을 보내줘."}]}

resume_type = "approve"

paused_result = approval_agent.invoke(request_message, approval_config)
print(paused_result["__interrupt__"])

resumed_result = approval_agent.invoke(
    Command(resume={"decisions": [{
            "type": resume_type
        }]
    }),
    approval_config
)

## 3. 멀티에이전트 - 나눌 때와 나누지 말아야 할 때
* 나누는 기준 = 역할 경계가 분명한가 (Tool, 데이터, 규칙이 다른가)
* 하위 에이전트를 부르는 함수를 @tool로 -> supervisor는 최종 답만 받는다 (컨텍스트 격리)
* hop마다 LLM 호출이 는다. 과한 위임은 느리고 추적이 어렵다
* 경계가 약하면 단일 에이전트 + 좋은 Tool 설계가 낫다

In [ ]:
MODEL = 'gpt-4o-mini'
lf_config = {}

@tool
def search_policy(query: str) -> str:
    """쇼핑몰 배송, 교환, 환불 규정을 조회합니다."""
    if "환불" in query or "반품" in query:
        return "단순 변심 환불은 상품 수령 후 7일 이내에 신청해야 하며, 반품 배송비는 고객 부담입니다."
    
    if "교환" in query:
        return "상품 수령 후 7일 이내에 교환을 요청할 수 있습니다."
    
    return "규정에서 관련 정보를 찾지 못했습니다."

@tool
def lookup_schedule(query: str) -> str:
    """배송 일정과 고객 안내 일정을 조회합니다."""
    if "주말" in query or "배송" in query:
        return "주말 주문은 월요일부터 순차 출고되며, 도서산간 지역은 1~2일 더 걸릴 수 있습니다."
    
    return "일정 정보가 없습니다."

policy_agent = create_agent(
    model=MODEL, 
    tools=[search_policy],
    system_prompt="쇼핑몰 규정 관련 질문만 답하는 정책 전문가입니다."
)
    
schedule_agent = create_agent(
    model=MODEL, 
    tools=[lookup_schedule],
    system_prompt="배송 일정 관련 질문만 답하는 운영 일정 전문가입니다."
)

@tool
def ask_policy_agent(question: str) -> str:
    """규정 관련 질문을 정책 에이전트에게 위임합니다"""
    result = policy_agent.invoke({
        "messages": [{
            "role": "user",
            "content": question
        }]
    }, lf_config)
    return result["messages"][-1].content

@tool
def ask_schedule_agent(question: str) -> str:
    """규정 관련 질문을 정책 에이전트에게 위임합니다"""
    result = policy_agent.invoke({
        "messages": [{
            "role": "user",
            "content": question
        }]
    }, lf_config)
    return result["messages"][-1].content

## 4. Agentic RAG - 같은 질문, 3가지 구조

1. Naive RAG
    * 검색 -> 생성, 항상
    * 코드가 정함
    * 검색이 부족해도 그냥 답함
    * 가장 싸고 빠름
    * 추적 쉬움

2. 검색 Tool + 단일 에이전트
    * 모델이 검색 필요 여부 판단
    * 모델이 정함
    * 검색이 부족할 때 모델이 다시 검색할 수도, 안 할 수도 있음
    * 비용/속도 중간
    * 추적 어려움

3. LangGraph Agentic RAG
    * 검색 -> 평가 -> (답변|재작성 -> 재검색|안전종료)
    * 코드가 지도하고 LLM이 평가
    * 검색이 부족하면 재작성 후 재검색, 한도 넘으면 "모르겠다"로 종료
    * 가장 비싸고 느림
    * 추적 쉬움(State의 relevant, retry_count)

==> ```Agentic RAG```?
* 새 검색 기법이 아닌 RAG 흐름 안에 판단을 넣은 구조
* 검색이 필요한가, 어떤 검색어를 쓸까, 결과가 충분한가, 언제 답을 만들까 - 이 중 일부라도 실행 중에 판단하면 agentic RAG
* "도구 사용, 상태 관리, 흐름 제어가 합쳐진 패턴"

==> 주의
* 고정된 검색 흐름만으로 품질이 이미 충분하거나, 응답 속도가 중요하거나, 비용 제한이 있으면 먼저 Naive RAG 검토
* 추가 판단이 필요 없는 상황에서 Agentic RAG를 쓰면 성능 개선 없이 비용과 지연만 늘어남
* 검색을 더 똑똑하게 보다 "정말 추가 의사결정이 필요한가?"를 먼저 묻는 것
* Hit@K로 Naive RAG가 이미 0.9를 넘는다면 Agentic으로 갈 이유가 없음

In [33]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from typing import NotRequired, TypedDict
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langgraph.graph import StateGraph, START, END

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)
lf_config = {}


# 준비 - 관련 문서 4개 + 무관한 문서 4개
docs = [
    Document(page_content="무료배송 주문에서 일부 상품을 반품해 최종 구매금액이 50,000원 미만이 되면 최초 배송비 3,000원을 환불액에서 차감한다.",
             metadata={"source": "partial_return_policy"}),
    Document(page_content="고객이 오배송 또는 상품 불량으로 환불을 요청하면 쇼핑몰이 왕복 배송비를 부담한다. …",
             metadata={"source": "defect_refund_policy"}),
    Document(page_content="사용한 쿠폰은 주문 취소 후 자동 재발급되지 않을 수 있다. …", metadata={"source": "coupon_reissue_policy"}),
    Document(page_content="도서산간 지역은 일반 배송보다 1~2일 더 걸릴 수 있다. …", metadata={"source": "remote_area_shipping_policy"}),
    # ↓ 쇼핑몰과 무관한 사내 문서 4개 (검색이 제대로 걸러내는지 보려고 섞어 둠)
    Document(page_content="사내 회의실 예약은 그룹웨어 캘린더에서 신청하며, …", metadata={"source": "meeting_room_policy"}),
    Document(page_content="임직원 노트북 반출은 보안 포털에서 사전 신청해야 하며, …", metadata={"source": "device_security_policy"}),
    Document(page_content="점심 식대 지원은 평일 근무일 기준으로 제공되며, …", metadata={"source": "meal_expense_policy"}),
    Document(page_content="신규 입사자는 입사 첫 주에 개인정보 보호 교육과 …", metadata={"source": "onboarding_policy"}),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)

# 검색 Tool을 붙인 단일 에이전트
@tool
def retrieve_docs(query: str) -> str:
    """내부 문서에서 쇼핑몰 고객지원 질문과 관련된 내용을 검색합니다"""
    found = vectorstore.similarity_search(query, k=3)
    return "\n\n".join(f"[source={doc.metadata.get('source')}] {doc.page_content}" for doc in found)

rag_agent = create_agent(
    model=MODEL,
    tools=[retrieve_docs],
    system_prompt="문서 기반 질문에는 retrieve_docs 도구를 사용하고, 검색 결과를 근거로 답변하세요.",
)

rag_result = rag_agent.invoke({"messages": [{
    "role": "user",
    "content": "무료배송으로 6만원을 주문했는데 2만원짜리 상품만 반품하면 최초 배송비가 차감되나?"
}]}, lf_config)

# LangGraph로 - 평가 노드가 핵심
class RAGState(TypedDict):
    question: str
    search_query: NotRequired[str]
    documents: NotRequired[list]
    relevant: NotRequired[bool]
    answer: NotRequired[str]
    retry_count: int
    max_retries: int

class RelevanceCheck(BaseModel):
    relevant: bool = Field(description="검색 문서가 질문과 직접 관련이 있으면 true")

relevance_model = llm.with_structured_output(RelevanceCheck)

# 노드와 라우터
def rewrite_query(state: RAGState):
    prompt = f"다음 질문을 검색 친화적으로 다시 써주세요: {state['question']}"
    rewritten = llm.invoke(prompt, config=lf_config)
    return {
        "search_query": rewritten.content,
        "retry_count": state.get("retry_count", 0) + 1
    }

def retrieve(state: RAGState):
    query = state.get("search_query") or state["question"]
    return {"documents": vectorstore.similarity_search(query, k=3)}

def grade_document(state: RAGState):
    joined = "\n\n".join(doc.page_content for doc in state['documents'])
    graded = relevance_model.invoke(
        f"질문: {state['question']} \n\n문서: \n{joined}\n\n이 문서들이 질문에 충분히 관련 있나요?"
    )
    return {"relevant": graded.relevant}

def generate_answer(state: RAGState):
    joined = "\n\n".join(doc.page_content for doc in state['documents'])
    answer = llm.invoke(
        f"질문: {state['question']}\n\n참고 문서: \n{joined}\n\n위 문서를 기반으로 한국어로 답하세요.", 
        config=lf_config
    )
    return {"answer": answer.content}

def fallback_answer(state: RAGState):
    return {
        "answer": "검색 결과가 충분하지 않아 추가 확인이 필요합니다. 검색어를 더 구체적으로 바꾸거나 사람이 문서를 확인하세요."
    }

def route_after_grade(state: RAGState):
    if state["relevant"]:
        return "답변 생성"
    if state.get("retry_count", 0) >= state.get("max_retries", 1):
        return "안전 종료"
    return "검색어 재작성"

# 연걸
builder = StateGraph(RAGState)

builder.add_node("문서 검색", retrieve)
builder.add_node("관련성 평가", grade_document)
builder.add_node("검색어 재작성", rewrite_query)
builder.add_node("답변 생성", generate_answer)
builder.add_node("안전 종료", fallback_answer)

builder.add_edge(START, "문서 검색")
builder.add_edge("문서 검색", "관련성 평가")
builder.add_conditional_edges(
    "관련성 평가",
    route_after_grade,
    {
        "답변 생성": "답변 생성",
        "검색어 재작성": "검색어 재작성",
        "안전 종료": "안전 종료"
    }
)
builder.add_edge("검색어 재작성", "문서 검색")
builder.add_edge("답변 생성", END)
builder.add_edge("안전 종료", END)

graph = builder.compile()

result = graph.invoke({
    "question": "무료배송으로 6만원을 주문했는데 2만원짜리 상품만 반품하면 최초 배송비가 차감되나?",
    # "question": "포인트 적립률은 몇 %야?",
    # "question": "회의실 예약은 어디서 해?",
    "retry_count": 0,
    "max_retries": 1
})

print(result.get('search_query') or result["question"])
print(result["relevant"])
print(result["answer"])

무료배송으로 6만원을 주문했는데 2만원짜리 상품만 반품하면 최초 배송비가 차감되나?
True
네, 무료배송으로 6만원을 주문하셨고 2만원짜리 상품만 반품하신다면, 최종 구매금액이 50,000원 미만이 되므로 최초 배송비 3,000원이 환불액에서 차감됩니다. 따라서 반품 시 주의하시기 바랍니다.


[요약]
* Naive(코드가 결정) -> Tool 에이전트(모델이 결정) -> 그래프(코드가 지도, LLM이 평가)
* Agentic RAG는 "RAG 흐름에 판단을 넣는 구조". 판단이 필요 없으면 Naive가 나음
* "안전 종료" 출구: 부족하면 지어내지 말고 멈춘다
* 횟수를 어디서 세는지(검색 vs 재작성)에 따라 한도의 뜻이 달라짐

## 5. 관측성과 테스트 - Langfuse와 가짜 모델

* Tool 호출, HITL, 분기, 재시도가 들어가면 최종 답변만 보고는 무엇이 일어났는지 알 수 없음
* 테스트: 원하는 흐름이 재현되는가
* 관측성: 실제 실해잉 어떻게 흘렀는가

### 5-1. stream이 있는데 왜 Langfuse인지?

* stream(stream_mode="updates")로 경로 확인 가능 -> 지금 눈앞의 실행 한 번
* Langfuse를 쓰면 실행이 끝난 뒤에도 기록(trace)이 남고, 여러 실행을 검색/비교하고, 팀원이 같은 화면에서 함께 볼 수 있음

### 5-2. 연결 순서
1. Organization 만들기
    * Langfuse는 프로젝트를 조직 단위로 관리합니다

2. Project 만들기
    * trace가 쌓일 곳. 실습용과 운영용을 나눠두면 나중에 구분하기 편함

3. API Key 확인
    * Project Settings에서 Public Key(프로젝트 식별용)와 Secret Key(서버 인증용)

4. .env에 키 넣기

5. CallbackHandler 연결
    * lf_config에 넣으면 끝

6. trace 읽기

In [ ]:
from dotenv import load_dotenv
from langfuse.langchain import CallbackHandler

load_dotenv()

langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} 

### 5-4. GenericFakeChatModel - 돈 안 드는 결정론적 테스트

In [34]:
from langchain_core.language_models import GenericFakeChatModel
from langchain.messages import AIMessage
from langchain.agents import create_agent

fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(content="배송 지연 문의는 주문 상태 확인 후 예상 도착일을 안내합니다.")
    ])
)

fake_agent = create_agent(
    model=fake_model,       # 실제 API 대신 가짜 모델
    tools=[],
    system_prompt="배송 지연 문의에 한 문장으로 답하는 에이전트입니다.",
)
fake_result = fake_agent.invoke({"messages": [{"role": "user", "content": "주문 ORD-1001 배송 지연 안내를 짧게 제안해줘."}]})
print(fake_result["messages"][-1].content)

배송 지연 문의는 주문 상태 확인 후 예상 도착일을 안내합니다.


### 5-5. 운영 전 체크리스트
* ```Architecture```: Agent, Graph, Multi-agent, RAG 구조가 과하지 않은가?

* ```Tool 설계```: 이름, docstring, 입력 스키마가 명확한가?

* ```HITL```: 위험 Tool에 승인이 적용되어 있는가?

* ```Memory```: thread_id와 상태 범위가 분리되었는가?

* ```Graph```: 분기 또는 상태 관리가 1개 이상 포함되어 있는가?

* ```RAG```: 검색 결과가 부족할 때 fallback 또는 재검색 제한이 있는가?

* ```Observability```: trace에서 Tool 호출과 중단/재개 흐름을 확인했는가?

* ```Test```: 최소한의 결정론적 스모크 테스트가 있는가?

[요약]
* stream은 지금 한 번, Langfuse는 끝난 뒤에도 남고 비교할 수 있는 기록
* lf_config = {"callbacks": [CallbackHandler()]}
* 그래프를 invoke할 때도 config를 넘겨야 한 trace로 묶임
* trace에서 가장 먼저 볼 것은 호출 횟수
* GenericFakeChatModel로 흐름(middleware, 라우터, Edge)을 돈 안 들이고 테스트

# [치트시트]
* 작업 -> 판정 -> (통과 | 고쳐서 다시) + 횟수 제한

In [36]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

MAX_TRIES = 3

# 1. State - 칸마다 "쌓을까(Annotated) / 바꿀까(그냥)"를 정함
class TestState(TypedDict):
    question: str
    query: str                                  # 바꾸기
    results: list[dict]                         # 바꾸기 (재검색하면 새 결과로)
    ok: bool                                    # 판정 결과 (사실만)
    reason: str                                 # 판정 이유 (고치기 노드가 읽음)
    tries: int                                  # 횟수
    answer: str

# 2. 노드 - State를 받아 바꾼 칸만 돌려줌
def work(state: TestState) -> dict:
    query = state.get("query") or state["question"]
    tries = state.get("tries", 0) + 1               # 작업하는 곳에서 셈
    results = [{"n": i} for i in range(tries)]      # 실제로는 검색, LLM 호출 (여기선 시도마다 1개씩 늘어나는 가짜)
    return {"query": query, "results": results, "tries": tries, "log": [f"work#{tries}"]}

def judge(state: TestState) -> dict:
    ok = len(state["results"]) >= 3
    return {"ok": ok, "reason": "충분" if ok else "부족", "log": ["judge"]}

def fix(state: TestState) -> dict:
    return {"query": state["query"] + " 구체적으로", "log": ["fix"]}

def finish(state: TestState) -> dict:
    return {"answer": "…", "log": ["finish"]}

# 3. 라우터 - 읽기만, 문자열 반환. 조건 순서가 규칙
def route(state: TestState) -> str:
    if state["ok"]:
        return "finish"

    if state["tries"] >= MAX_TRIES:
        return "finish"
    
    return "fix"

# 4. 조립
build = StateGraph(TestState)

for name, fn in [("work", work), ("judge", judge), ("fix", fix), ("finish", finish)]:
    builder.add_node(name, fn)

builder.add_edge(START, "work")
builder.add_edge("work", "judge")
builder.add_conditional_edges(
    "judge",
    route,
    {
        "finish": "finish",
        "fix": "fix",
    }
)
builder.add_edge("fix", "work")
builder.add_edge("finish", END)

app = builder.compile()

# 5. 실행 - 한도를 한번 더 걸어두고, 경로는 stream으로 확인
init = {
    "question": "...",
    "query": "",
    "results": [],
    "ok": False,
    "reason": "",
    "tries": 0,
    "log": [],
    "answer": ""
}

for ev in app.stream(init, stream_mode="updates", config={"recursion_limit": 30}):
    print(ev)

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


{'work': {'query': '...', 'results': [{'n': 0}], 'tries': 1}}
{'문서 검색': {'documents': [Document(id='c00da608-8572-4899-a5d4-c3dd07c416af', metadata={'source': 'device_security_policy'}, page_content='임직원 노트북 반출은 보안 포털에서 사전 신청해야 하며, …'), Document(id='6a611be9-e6fd-4ba9-a680-23622308aa5b', metadata={'source': 'meeting_room_policy'}, page_content='사내 회의실 예약은 그룹웨어 캘린더에서 신청하며, …'), Document(id='5b6c0137-f632-462c-9f8f-a17c7a344008', metadata={'source': 'coupon_reissue_policy'}, page_content='사용한 쿠폰은 주문 취소 후 자동 재발급되지 않을 수 있다. …')]}}
{'judge': {'ok': False, 'reason': '부족'}}
{'관련성 평가': {'relevant': False}}
{'fix': {'query': '... 구체적으로'}}
{'검색어 재작성': {'search_query': '물론입니다! 질문을 검색 친화적으로 바꾸고 싶으신 내용을 알려주시면, 그에 맞게 수정해드리겠습니다. 어떤 질문인지 말씀해 주세요.', 'retry_count': 1}}
{'work': {'query': '... 구체적으로', 'results': [{'n': 0}, {'n': 1}], 'tries': 2}}
{'문서 검색': {'documents': [Document(id='9f96aa29-093e-4d2f-b051-e4ca1d5d7fb6', metadata={'source': 'defect_refund_policy'}, page_content='고객이 오배송 또는 상품 불량으로 환불을 요

"우리 실험실 프로토콜에서 스페로이드 배양 조건 찾아서, 최근 논문 동향이랑 같이 정리해줘."
   ↓
[5권] 그래프 시작 · State = {question, protocol_docs, papers, relevant, tries, report}
   ├─ 노드 A: 프로토콜 검색   [3권] 내부 문서 retriever             ┐ [5권] Fan-out
   └─ 노드 B: 논문 검색       [5권 07] Tavily (topic 없이 학술 키워드) ┘      (병렬)
   ↓ [5권] Fan-in
노드 C: 관련성 평가           [4·5권] with_structured_output(bool)
   ├─ 부족 · 횟수 남음 → 검색어 재작성 [2권] → A·B로   (루프)
   ├─ 한도 도달        → 안전 종료 "사람이 확인하세요"
   └─ 충분             ↓
노드 D: 요약·리포트            [1권] prompt | llm | parser, [2권] 근거 제한 규칙
   ↓
[5권] 환자·참여자 정보가 섞였으면 PIIMiddleware(apply_to_tool_results)
[3·5권] Langfuse trace로 호출 횟수·검색어·시간 확인